<a href="https://colab.research.google.com/github/Lee-hosuk/game-agent/blob/main/llm-tictactoe/llm_tictactoe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⭕❌ LLM 틱택토 에이전트

이번에는 강화학습이 아니라 **LLM(언어 모델)에게 판을 글로 보여주고 다음 수를 고르게** 합니다.

| | 강화학습 (CartPole, FrozenLake) | LLM 에이전트 (이번) |
|---|---|---|
| 학습 | 수천 판 플레이하며 학습 | **학습 없음** (이미 학습된 모델 사용) |
| obs | 숫자 | **글** (판 상태를 텍스트로) |
| 행동 고르기 | 신경망/표의 점수 | **프롬프트 → LLM 답변 → 파싱** |
| 실력을 올리는 방법 | 더 학습 | **프롬프트 개선** |

코랩에 내장된 `google.colab.ai`를 사용하므로 **API 키가 필요 없습니다.**

## 1. 틱택토 게임 만들기

칸 번호는 LLM이 이해하기 쉽게 1~9로 씁니다.
```
 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9
```

In [ ]:
import random
import re
import time

WIN_LINES = [(0,1,2), (3,4,5), (6,7,8),   # 가로
             (0,3,6), (1,4,7), (2,5,8),   # 세로
             (0,4,8), (2,4,6)]            # 대각선

def new_board():
    return [" "] * 9                       # 빈칸 9개

def legal_moves(board):
    return [i + 1 for i in range(9) if board[i] == " "]   # 둘 수 있는 칸 번호 (1~9)

def winner(board):
    for a, b, c in WIN_LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]                # "X" 또는 "O"
    if " " not in board:
        return "draw"
    return None                            # 아직 진행 중

def board_to_text(board):
    """판을 글로 바꾸기. 빈칸은 칸 번호로 표시해서 LLM이 고르기 쉽게 함"""
    cells = [board[i] if board[i] != " " else str(i + 1) for i in range(9)]
    rows = [f" {cells[r*3]} | {cells[r*3+1]} | {cells[r*3+2]}" for r in range(3)]
    return "\n---+---+---\n".join(rows)

b = new_board()
b[0], b[4] = "X", "O"
print(board_to_text(b))
print("둘 수 있는 칸:", legal_moves(b))

## 2. 비교용 상대 만들기

LLM 실력을 재려면 기준이 필요합니다.
- **랜덤 봇**: 아무 칸에나 둠 → 가장 약한 기준
- **미니맥스 봇**: 모든 경우의 수를 끝까지 계산 → **절대 지지 않는 완벽한 플레이어**. 틱택토는 둘 다 완벽하면 무조건 무승부입니다.

In [ ]:
def random_bot(board, me):
    return random.choice(legal_moves(board))

def minimax(board, me, turn):
    """turn 차례일 때, me 입장에서의 점수 (+1 승, 0 무, -1 패)"""
    w = winner(board)
    if w == me: return 1
    if w == "draw": return 0
    if w is not None: return -1
    other = "O" if turn == "X" else "X"
    scores = []
    for m in legal_moves(board):
        board[m - 1] = turn
        scores.append(minimax(board, me, other))
        board[m - 1] = " "
    return max(scores) if turn == me else min(scores)

def minimax_bot(board, me):
    other = "O" if me == "X" else "X"
    best, best_moves = -2, []
    for m in legal_moves(board):
        board[m - 1] = me
        s = minimax(board, me, other)
        board[m - 1] = " "
        if s > best:
            best, best_moves = s, [m]
        elif s == best:
            best_moves.append(m)
    return random.choice(best_moves)       # 같은 점수면 랜덤 (매번 같은 게임이 되지 않게)

## 3. 대전 함수
두 플레이어(함수)를 받아서 한 판을 진행합니다. X가 먼저 둡니다.

In [ ]:
def play_game(player_x, player_o, verbose=True):
    board = new_board()
    players = {"X": player_x, "O": player_o}
    turn = "X"
    while winner(board) is None:
        move = players[turn](board, turn)
        if move not in legal_moves(board):          # 안전장치
            move = random.choice(legal_moves(board))
        board[move - 1] = turn
        if verbose:
            print(f"[{turn}] → {move}번 칸\n{board_to_text(board)}\n")
        turn = "O" if turn == "X" else "X"
    result = winner(board)
    if verbose:
        print("결과:", "무승부" if result == "draw" else f"{result} 승리!")
    return result

random.seed(0)
play_game(minimax_bot, random_bot)

## 4. LLM 연결하기

`google.colab.ai`는 코랩에서 무료로 Gemini 모델을 쓸 수 있게 해줍니다.
사용 가능한 모델은 계정마다 다를 수 있으니 먼저 목록을 확인합니다.

In [ ]:
from google.colab import ai

print(ai.list_models())

In [ ]:
MODEL = None   # None이면 기본 모델. 위 목록에서 골라 넣어도 됨 (예: "google/gemini-2.5-flash")

def ask_llm(prompt):
    if MODEL:
        return ai.generate_text(prompt, model_name=MODEL)
    return ai.generate_text(prompt)

print(ask_llm("틱택토 규칙을 한 문장으로 설명해줘."))

## 5. LLM 에이전트 만들기 ⭐ (핵심)

LLM 에이전트의 흐름은 강화학습의 obs → 행동과 같습니다. 다만 모든 게 **글**입니다.
```
판 상태 → [프롬프트로 변환] → LLM → 답변 글 → [숫자 뽑기] → 행동
```

**LLM 에이전트의 흔한 문제**: 이미 찬 칸을 고르거나, 숫자를 이상하게 답하기도 합니다.
그래서 **잘못된 수면 이유를 알려주고 다시 묻는** 처리가 필요합니다. 3번 실패하면 랜덤으로 둡니다.

In [ ]:
BASIC_PROMPT = """너는 틱택토를 두고 있어. 너는 '{me}'이고 상대는 '{other}'야.
가로, 세로, 대각선으로 같은 기호 3개를 먼저 놓으면 이겨.

현재 판 (숫자는 비어 있는 칸 번호):
{board}

둘 수 있는 칸: {legal}

어디에 둘지 짧게 이유를 말하고, 마지막 줄에 반드시 'MOVE: 칸번호' 형식으로 답해."""

stats = {"calls": 0, "invalid": 0, "fallback": 0}   # LLM이 몇 번 실수했는지 기록

def parse_move(text):
    """답변에서 'MOVE: 5' 형식의 숫자 뽑기. 없으면 마지막에 나온 1~9 숫자"""
    m = re.findall(r"MOVE\s*[:：]\s*([1-9])", text, re.IGNORECASE)
    if m:
        return int(m[-1])
    nums = re.findall(r"[1-9]", text)
    return int(nums[-1]) if nums else None

def make_llm_player(prompt_template, show_reason=True, max_tries=3):
    def llm_player(board, me):
        other = "O" if me == "X" else "X"
        prompt = prompt_template.format(me=me, other=other,
                                        board=board_to_text(board),
                                        legal=legal_moves(board))
        for attempt in range(max_tries):
            stats["calls"] += 1
            try:
                answer = ask_llm(prompt)
            except Exception as e:                  # 사용량 제한 등 오류 → 잠깐 쉬고 다시
                print("LLM 오류:", e)
                time.sleep(5)
                continue
            move = parse_move(answer)
            if move in legal_moves(board):
                if show_reason:
                    print(f"🤖 LLM({me}) 생각: {answer.strip()}\n")
                return move
            stats["invalid"] += 1
            prompt += f"\n\n'{move}'은(는) 둘 수 없는 칸이야. 다음 중에서 골라: {legal_moves(board)}"
        stats["fallback"] += 1
        return random.choice(legal_moves(board))
    return llm_player

llm_basic = make_llm_player(BASIC_PROMPT)

## 6. LLM vs 랜덤 봇 한 판
LLM이 X(선공)입니다. LLM이 왜 그 칸을 골랐는지도 같이 출력됩니다.

In [ ]:
random.seed(1)
play_game(llm_basic, random_bot)
print(stats)

## 7. 여러 판 대전으로 실력 측정하기

한 판은 운이 섞이니 여러 판을 둡니다. 선공/후공도 번갈아 바꿉니다.
⏱️ LLM을 수십 번 호출하므로 몇 분 걸릴 수 있습니다. 무료 사용량 제한에 걸리면 `n_games`를 줄여주세요.

In [ ]:
def tournament(llm_player, opponent, n_games=6, label=""):
    record = {"승": 0, "무": 0, "패": 0}
    for g in range(n_games):
        llm_first = (g % 2 == 0)
        if llm_first:
            result, llm_mark = play_game(llm_player, opponent, verbose=False), "X"
        else:
            result, llm_mark = play_game(opponent, llm_player, verbose=False), "O"
        if result == "draw":
            record["무"] += 1
        elif result == llm_mark:
            record["승"] += 1
        else:
            record["패"] += 1
    print(f"{label:<28} 승 {record['승']} / 무 {record['무']} / 패 {record['패']}")
    return record

quiet_basic = make_llm_player(BASIC_PROMPT, show_reason=False)
random.seed(0)
stats.update(calls=0, invalid=0, fallback=0)
r1 = tournament(quiet_basic, random_bot,  label="기본 프롬프트 vs 랜덤")
r2 = tournament(quiet_basic, minimax_bot, label="기본 프롬프트 vs 미니맥스")
print("LLM 호출:", stats["calls"], "| 잘못된 수:", stats["invalid"], "| 랜덤으로 대체:", stats["fallback"])

**볼 포인트**
- 미니맥스는 절대 지지 않으므로, LLM의 최선은 **무승부**입니다. 지는 판이 있다면 LLM이 실수를 한 겁니다.
- 랜덤 봇 상대로도 지거나 비긴다면, LLM이 "이길 수 있는 수"나 "막아야 하는 수"를 놓친 겁니다.

## 8. 프롬프트 개선하기 🧪

강화학습에서는 실력을 올리려면 더 학습해야 했지만, LLM 에이전트는 **프롬프트만 바꿔서** 실력을 올릴 수 있습니다.
전략을 알려주고, 판을 단계별로 분석하게 해봅니다.

In [ ]:
STRATEGY_PROMPT = """너는 틱택토 고수야. 너는 '{me}'이고 상대는 '{other}'야.
가로, 세로, 대각선으로 같은 기호 3개를 먼저 놓으면 이겨.

현재 판 (숫자는 비어 있는 칸 번호):
{board}

둘 수 있는 칸: {legal}

아래 순서대로 하나씩 확인해:
1. 내가 지금 두면 바로 이기는 칸이 있나? 있으면 거기에 둬.
2. 상대가 다음 차례에 이기는 칸이 있나? 있으면 거기를 막아.
3. 한 번에 두 줄을 동시에 위협하는 칸(포크)이 있나? 있으면 거기에 둬.
4. 상대의 포크를 막아야 하나?
5. 가운데(5)가 비어 있으면 가운데.
6. 상대가 모서리에 있으면 반대쪽 모서리.
7. 빈 모서리(1, 3, 7, 9).
8. 남은 칸.

이길 수 있는 줄(가로 3개, 세로 3개, 대각선 2개)을 하나씩 확인한 뒤,
마지막 줄에 반드시 'MOVE: 칸번호' 형식으로 답해."""

quiet_strategy = make_llm_player(STRATEGY_PROMPT, show_reason=False)
random.seed(0)
stats.update(calls=0, invalid=0, fallback=0)
r3 = tournament(quiet_strategy, random_bot,  label="전략 프롬프트 vs 랜덤")
r4 = tournament(quiet_strategy, minimax_bot, label="전략 프롬프트 vs 미니맥스")
print("LLM 호출:", stats["calls"], "| 잘못된 수:", stats["invalid"], "| 랜덤으로 대체:", stats["fallback"])

## 9. 결과 비교

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ["basic\nvs random", "basic\nvs minimax", "strategy\nvs random", "strategy\nvs minimax"]
records = [r1, r2, r3, r4]
win  = np.array([r["승"] for r in records])
draw = np.array([r["무"] for r in records])
loss = np.array([r["패"] for r in records])

plt.figure(figsize=(8, 4))
plt.bar(labels, win, label="win", color="#4caf50")
plt.bar(labels, draw, bottom=win, label="draw", color="#9e9e9e")
plt.bar(labels, loss, bottom=win + draw, label="loss", color="#f44336")
plt.ylabel("games")
plt.title("LLM Tic-Tac-Toe: basic vs strategy prompt")
plt.legend()
plt.savefig("ttt_results.png", dpi=100, bbox_inches="tight")
plt.show()

## 10. 직접 LLM과 대결하기 🎮
버튼을 눌러서 LLM과 둬보세요. 기본은 당신이 X(선공)이고, 마지막 줄에서 `human_mark="O"`로 바꾸면 후공으로 둘 수 있습니다.
한 판 더 하려면 셀을 다시 실행하세요.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def play_vs_llm(llm_player, human_mark="X"):
    llm_mark = "O" if human_mark == "X" else "X"
    board = new_board()

    # 3x3 버튼 판
    buttons = [widgets.Button(layout=widgets.Layout(width="70px", height="70px")) for _ in range(9)]
    status = widgets.HTML()
    log = widgets.Output()          # LLM의 생각이 여기에 출력됨

    def refresh():
        over = winner(board) is not None
        for i, b in enumerate(buttons):
            b.description = board[i] if board[i] != " " else str(i + 1)
            b.disabled = board[i] != " " or over
            b.button_style = {"X": "info", "O": "warning"}.get(board[i], "")

    def check_end():
        w = winner(board)
        if w is None:
            return False
        if w == "draw":
            status.value = "<h3>🤝 무승부!</h3>"
        elif w == human_mark:
            status.value = "<h3>🎉 당신이 이겼어요!</h3>"
        else:
            status.value = "<h3>🤖 LLM이 이겼어요!</h3>"
        refresh()
        return True

    def llm_turn():
        status.value = "<h3>🤖 LLM이 생각 중...</h3>"
        for b in buttons:
            b.disabled = True
        with log:
            clear_output()
            move = llm_player(board, llm_mark)
        board[move - 1] = llm_mark
        refresh()
        if not check_end():
            status.value = f"<h3>당신({human_mark}) 차례예요. 칸을 클릭하세요.</h3>"

    def on_click(b):
        i = buttons.index(b)
        if board[i] != " " or winner(board) is not None:
            return
        board[i] = human_mark
        refresh()
        if not check_end():
            llm_turn()

    for b in buttons:
        b.on_click(on_click)

    grid = widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(3, 75px)"))
    display(widgets.VBox([status, grid, log]))
    refresh()
    if human_mark == "O":
        llm_turn()                  # 내가 후공이면 LLM이 먼저 둠
    else:
        status.value = f"<h3>당신({human_mark}) 차례예요. 칸을 클릭하세요.</h3>"

play_vs_llm(make_llm_player(STRATEGY_PROMPT), human_mark="X")   # 후공으로 하려면 human_mark="O"


## 🔥 도전 과제
1. **모델 바꿔보기**: `MODEL`을 4번 셀의 목록에 있는 다른 모델로 바꿔서 실력을 비교해보세요.
2. **판 표현 바꾸기**: `board_to_text`를 바꿔서 판을 다르게 보여주면 실력이 달라질까요? (예: 좌표 `(행, 열)` 방식, 줄별로 설명)
3. **잘못된 수 줄이기**: 잘못된 수가 자주 나온다면 프롬프트를 어떻게 바꾸면 될까요?
4. **LLM vs LLM**: 기본 프롬프트 LLM과 전략 프롬프트 LLM끼리 대결시켜보세요.
5. **강화학습과 비교**: FrozenLake처럼 Q-러닝으로 틱택토 에이전트를 학습시키면 LLM보다 강해질까요?